# 04 — Experimentos Controlados: Dados Brutos × Framework

Etapas 2, 4 e 5 do procedimento metodológico (Seção 3.4): aplicar o pipeline de normalização e balanceamento e
comparar a Rede Bayesiana treinada com **dados brutos** e com **dados tratados pelo framework**.

> **H1:** A utilização de um framework dedicado de normalização de dados melhora significativamente a acurácia e o
> Recall de modelos probabilísticos estruturados na predição de no-show.
>
> **H3:** Modelos probabilísticos alimentados com dados tratados por técnicas avançadas de escalonamento
> (RobustScaler, MinMaxScaler ou StandardScaler) e sobreamostragem estrutural (SMOTENC) apresentam desempenho
> estatisticamente superior, mensurado pelas métricas ROC-AUC e PR-AUC sob validação cruzada, aos modelos treinados
> com dados brutos.

## Desenho experimental

**Ablação em degraus.** Cada configuração acrescenta um componente do framework, para isolar o efeito de cada um:

| Config. | Dados | Escalonador | SMOTENC |
|---|---|---|---|
| **C0 bruto** (controle) | colunas originais, sem limpeza nem engenharia | — | — |
| C0 bruto + SMOTENC | colunas originais | Robust | ✓ |
| **C1 engenharia** | variáveis do notebook 02 (limpeza, histórico, clima corrigido) | — | — |
| C2 engenharia + escalonador | idem | MinMax / Standard / Robust | — |
| **C3 engenharia + SMOTENC** (framework completo) | idem | nenhum / MinMax / Standard / Robust | ✓ |

**Por que o controle não é "100% bruto".** Uma Rede Bayesiana discreta não aceita números contínuos, então todas as
configurações passam pela mesma discretização por quantis. O controle C0 é a base como chega: `age` errada, clima
com o artefato de "frio intenso", sem histórico e sem ID. Só saem a coluna de vazamento (`no_show_reason`) e as datas
cruas.

**Em todas as configurações e folds:**
1. o escalonador, o SMOTENC e a discretização são ajustados **só no treino**; o teste nunca é reamostrado;
2. a estrutura é aprendida no treino (Hill Climbing com AIC, a melhor configuração da H2) e reduzida ao Manto de
   Markov;
3. as CPTs são estimadas no treino (priori BDeu).

**Duas validações:**
- **Validação cruzada** 5 folds × 3 repetições, agrupada por paciente (15 pares para o Wilcoxon). É o desenho da H3.
- **Validação temporal:** treino até 2020 e teste em 2021–2022. É o cenário de produção, em que o modelo prevê o
  futuro. Ano e período ficam de fora, porque o futuro nunca tem um valor visto no treino.

O cálculo completo leva ~25 minutos e é feito por `scripts/run_h3.py`. Este notebook carrega os resultados de
`resultados/` (mude `RECALCULAR = True` para refazer tudo).

In [ ]:
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats

from noshow.data import TARGET
from noshow.evaluation import paired_wilcoxon, patient_groups, repeated_group_folds
from noshow.experiments import fit_predict
from noshow.features import CATEGORICAL, CONTINUOUS
from noshow.pipeline import SCALERS, raw_matrix, smotenc_resample

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RECALCULAR = False

if RECALCULAR:
    subprocess.run([sys.executable, str(RAIZ / "scripts" / "run_h3.py")], check=True,
                   env={**os.environ, "PYTHONHASHSEED": "0"})
CV = pd.read_csv(RAIZ / "resultados" / "h3_validacao_cruzada.csv")
TEMP = pd.read_csv(RAIZ / "resultados" / "h3_validacao_temporal.csv")
print(f"Validação cruzada: {CV.config.nunique()} configurações × {CV.groupby(['rep', 'fold']).ngroups} folds | "
      f"validação temporal: {len(TEMP)} configurações")

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
PCT = mtick.PercentFormatter(1, decimals=0)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
ORDEM = ["C0 bruto", "C0 bruto + SMOTENC (robust)", "C1 engenharia",
         "C3 engenharia + SMOTENC (sem escalonador)", "C3 engenharia + SMOTENC (minmax)",
         "C3 engenharia + SMOTENC (standard)", "C3 engenharia + SMOTENC (robust)"]
COR = {c: GRAY if c.startswith("C0") else BLUE if c.startswith("C1") else ORANGE for c in ORDEM}

## 1. Escalonamento sem SMOTENC: efeito nulo por construção (C2)

MinMax, Standard e Robust são transformações lineares crescentes. Elas preservam a **ordem** dos valores, logo os
quantis e as faixas da discretização caem nos mesmos registros. Para a Rede Bayesiana discreta, os dados escalonados
são **indistinguíveis** dos originais. A célula abaixo verifica isso em um fold real, comparando as probabilidades
previstas.

In [ ]:
df = pd.read_parquet(RAIZ / "data" / "processed" / "features.parquet")
y = df.no_show.to_numpy()
X = df[CATEGORICAL + CONTINUOUS]
_, _, tr, te = next(repeated_group_folds(y, patient_groups(df.patient_id), n_splits=5, n_repeats=1))

p_base, mb_base, _ = fit_predict(X.iloc[tr], y[tr], X.iloc[te], CATEGORICAL, CONTINUOUS)
print(f"C1 sem escalonador → Manto: {mb_base}")
for nome, Scaler in SCALERS.items():
    sc = Scaler().fit(X.iloc[tr][CONTINUOUS])                 # sklearn ignora NaN no ajuste e os preserva
    Xs = X.copy()
    Xs[CONTINUOUS] = sc.transform(X[CONTINUOUS])
    p, mb, _ = fit_predict(Xs.iloc[tr], y[tr], Xs.iloc[te], CATEGORICAL, CONTINUOUS)
    print(f"C2 {nome:8s} → Manto: {mb} | probabilidades idênticas a C1: {np.allclose(p, p_base)}")

Confirmado: **o escalonamento sozinho não altera nada na Rede Bayesiana discreta**. O único canal pelo qual o
escalonador pode influenciar o modelo é a **distância usada pelo SMOTENC** para escolher vizinhos (C3). Por isso,
C2 não foi repetida nos 15 folds.

## 2. Validação cruzada: resultados (H3)

In [ ]:
resumo = CV.groupby("config").agg(
    roc_auc=("roc_auc", "mean"), roc_dp=("roc_auc", "std"), pr_auc=("pr_auc", "mean"), pr_dp=("pr_auc", "std"),
    brier=("brier", "mean"), p_media=("p_media", "mean"), n_params=("n_params", "mean"),
).loc[ORDEM]
resumo["Manto mais frequente"] = CV.groupby("config").manto.agg(lambda s: s.value_counts().index[0]).loc[ORDEM]
resumo.style.format({"roc_auc": "{:.4f}", "roc_dp": "±{:.4f}", "pr_auc": "{:.4f}", "pr_dp": "±{:.4f}",
                     "brier": "{:.4f}", "p_media": "{:.1%}", "n_params": "{:,.0f}"}) \
    .highlight_max(subset=["roc_auc", "pr_auc"], color="#d6e6f8").highlight_min(subset=["brier"], color="#d6e6f8")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
ypos = {c: i for i, c in enumerate(ORDEM[::-1])}
rng = np.random.default_rng(0)
for ax, (met, titulo) in zip(axes, [("pr_auc", "PR-AUC (↑)"), ("roc_auc", "ROC-AUC (↑)"), ("brier", "Brier (↓)")]):
    for c in ORDEM:
        v = CV.loc[CV.config == c, met]
        ax.scatter(v, ypos[c] + rng.uniform(-0.15, 0.15, len(v)), s=14, color=COR[c], alpha=0.6, linewidths=0)
        ax.plot(v.mean(), ypos[c], "|", color=INK, ms=16, mew=2)
    ax.set_title(titulo); ax.grid(axis="y", visible=False)
axes[0].set_yticks(list(ypos.values()), list(ypos.keys()), fontsize=8)
fig.suptitle("Validação cruzada agrupada, 15 folds (pontos = folds, traço = média)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Testes de Wilcoxon pareados com correção de Holm.** A H3 envolve várias comparações ao mesmo tempo (três
escalonadores, duas métricas). Sem correção, a chance de algum *p* < 0,05 aparecer por acaso cresce. O método de
Holm ajusta os *p*-valores para manter o erro global em 5%.

In [ ]:
def holm(p):
    p = np.asarray(p); ordem = np.argsort(p); m = len(p)
    ajust = np.maximum.accumulate((m - np.arange(m)) * p[ordem]).clip(max=1)
    out = np.empty(m); out[ordem] = ajust
    return out

COMPARACOES = [(c, "C0 bruto") for c in ORDEM[1:]] + [(c, "C1 engenharia") for c in ORDEM[3:]]
linhas = []
for met in ["roc_auc", "pr_auc"]:
    piv = CV.pivot_table(index=["rep", "fold"], columns="config", values=met)
    for a, b in COMPARACOES:
        linhas.append({"comparação": f"{a}  ×  {b}", "métrica": met, **paired_wilcoxon(piv[a], piv[b])})
testes = pd.DataFrame(linhas)
testes["p_holm"] = holm(testes.p)
testes["conclusão"] = np.where(testes.p_holm >= 0.05, "sem diferença",
                               np.where(testes.diferença_mediana > 0, "SUPERIOR", "inferior"))
testes.set_index(["comparação", "métrica"]).style.format(
    {"diferença_mediana": "{:+.4f}", "p": "{:.4f}", "p_holm": "{:.4f}"}) \
    .map(lambda v: f"color: {BLUE}; font-weight: bold" if v == "SUPERIOR" else
         f"color: {ORANGE}; font-weight: bold" if v == "inferior" else "", subset=["conclusão"])

## 3. Por que o SMOTENC prejudica a rede

O SMOTENC cria um registro sintético de no-show a partir de um registro real e de um vizinho: as contínuas são
**interpoladas** e as categóricas recebem a **moda dos vizinhos**. Em agendamentos, isso quebra duas estruturas:

1. **O clima é um atributo do dia**, igual para todos os agendamentos daquela data. Interpolar a chuva entre dois
   pacientes de dias diferentes cria dias que **nunca existiram**: chuva de 7 mm/h com categoria "sem chuva", por
   exemplo.
2. **Combinações impossíveis viram "assinatura" de registro sintético.** A chuva numérica é interpolada, mas a
   categoria de chuva vem da moda dos vizinhos, então as duas deixam de ser coerentes. Essas combinações **só existem
   nos sintéticos**, e como todos os sintéticos são no-show, elas têm 100% de no-show no treino. O Hill Climbing
   aprende a reconhecer **registros fabricados**, e não faltas, e por isso a chuva entra no Manto.

A célula abaixo mede isso em um fold.

In [ ]:
Xtr = X.iloc[tr]
Xr, yr = smotenc_resample(Xtr, y[tr], CATEGORICAL, CONTINUOUS, scaler="robust")
sint = Xr.iloc[len(Xtr):]
print(f"Treino original: {len(Xtr):,} ({y[tr].mean():.1%} no-show) → reamostrado: {len(Xr):,} ({yr.mean():.0%}); "
      f"{len(sint):,} sintéticos")

# 1. combinações de clima que não existem em nenhum dia real
dias_reais = set(map(tuple, df[["max_temp_day", "max_rain_day"]].dropna().round(2).to_numpy()))
clima_sint = sint[["max_temp_day", "max_rain_day"]].dropna().round(2)
inexistentes = np.mean([tuple(r) not in dias_reais for r in clima_sint.to_numpy()])
print(f"Sintéticos com (temperatura, chuva) que não existem em nenhum dia real: {inexistentes:.0%}")

from noshow.weather import rain_category
incoerente = (rain_category(sint.max_rain_day) != sint.rain_intensity) & sint.max_rain_day.notna()
print(f"Sintéticos com rain_intensity incoerente com max_rain_day: {incoerente.mean():.0%} "
      f"(nos dados reais: {((rain_category(Xtr.max_rain_day) != Xtr.rain_intensity) & Xtr.max_rain_day.notna()).mean():.0%})")

In [ ]:
from noshow.features import Discretizer
sintetico = np.r_[np.zeros(len(Xtr), bool), np.ones(len(Xr) - len(Xtr), bool)]
D = Discretizer(columns=CONTINUOUS).fit(Xr).transform(Xr)
combo = D.max_rain_day + "  |  " + D.rain_intensity
impossivel = ~combo.isin(set(combo[~sintetico]))
print(f"Sintéticos com combinação (faixa de max_rain_day, rain_intensity) que não existe nos dados reais: "
      f"{impossivel[sintetico].mean():.0%}")
print(f"Taxa de no-show no treino reamostrado → combinação real: {yr[~impossivel].mean():.1%} | "
      f"combinação só-sintética: {yr[impossivel].mean():.1%} (n = {impossivel.sum():,})")
tab = pd.crosstab(combo[impossivel], yr[impossivel]).rename(columns={1: "no-show (todos sintéticos)"})
tab.sort_values(tab.columns[-1], ascending=False).head(6)

In [ ]:
freq = (CV.assign(chuva_no_manto=CV.manto.str.contains("rain"))
          .groupby("config").chuva_no_manto.mean().loc[ORDEM])
print("Fração dos folds em que alguma variável de chuva entra no Manto de Markov:")
print(freq.map("{:.0%}".format).to_string())

## 4. H1: acurácia e Recall

A H1 fala em **acurácia** e **Recall**, que dependem de um **limiar de decisão**: a partir de que probabilidade o
paciente é marcado como "vai faltar". Dois limiares são comparados:

- **0,5**, o padrão de um classificador;
- **a prevalência do treino** (~10%): alerta quando o risco do paciente está acima da média.

O ponto central: um modelo treinado com SMOTENC acredita que metade dos agendamentos são faltas, então suas
probabilidades ficam infladas e quase todos passam do limiar de 0,5. O Recall sobe **porque o modelo alerta mais**,
não porque discrimina melhor. Baixar o limiar de um modelo sem SMOTENC produz o mesmo efeito.

In [ ]:
cols = ["accuracy", "recall", "precision", "f1", "balanced_accuracy", "pct_alertas"]
lim = pd.concat({
    "limiar 0,5": CV.groupby("config")[[f"{c}@050" for c in cols]].mean().set_axis(cols, axis=1),
    "limiar = prevalência": CV.groupby("config")[[f"{c}@prev" for c in cols]].mean().set_axis(cols, axis=1),
}, axis=1).loc[ORDEM]
lim.style.format("{:.3f}").format("{:.1%}", subset=[(l, "pct_alertas") for l in ["limiar 0,5", "limiar = prevalência"]])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for c in ORDEM:
    for sufixo, marcador in [("050", "o"), ("prev", "s")]:
        sub = CV[CV.config == c]
        ax.scatter(sub[f"pct_alertas@{sufixo}"].mean(), sub[f"recall@{sufixo}"].mean(), color=COR[c], marker=marcador,
                   s=70, edgecolors="white", linewidths=1.5, zorder=3)
x = np.linspace(0, 1, 50)
ax.plot(x, x, color=GRAY, ls="--", lw=1)
ax.text(0.62, 0.55, "alertar ao acaso\n(recall = % alertado)", fontsize=8, color=INK)
ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel("% dos agendamentos marcados como risco"); ax.set_ylabel("Recall (faltas detectadas)")
ax.set_title("Recall × volume de alertas")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=GRAY, label="C0 bruto"),
                   Line2D([], [], marker="o", ls="", color=BLUE, label="C1 engenharia"),
                   Line2D([], [], marker="o", ls="", color=ORANGE, label="C3 + SMOTENC"),
                   Line2D([], [], marker="o", ls="", color=INK, mfc="white", label="limiar 0,5"),
                   Line2D([], [], marker="s", ls="", color=INK, mfc="white", label="limiar = prevalência")],
          frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

Todos os pontos ficam pouco acima da diagonal: o Recall que cada modelo atinge é, em grande parte, o volume de
alertas que ele dispara. A pergunta útil para o serviço não é "qual modelo tem mais Recall", e sim **quantas faltas
são detectadas para um mesmo volume de alertas**, que é exatamente o que a PR-AUC mede.

## 5. Validação temporal: o modelo prevê o futuro?

Treino com 2016–2020 e teste com 2021–2022 (fora: ano e período). É o cenário mais próximo de produção. Aqui os
dois critérios de estrutura são comparados: o AIC venceu na validação cruzada aleatória (notebook 03), mas um
critério mais permissivo pode estar aprendendo padrões de calendário que não se repetem no futuro.

In [ ]:
tab = TEMP.set_index(["criterio", "config"])[["roc_auc", "pr_auc", "brier", "p_media", "n_params", "manto"]]
tab = tab.reindex(pd.MultiIndex.from_product([["AIC", "BIC"], ORDEM]))
tab.style.format({"roc_auc": "{:.4f}", "pr_auc": "{:.4f}", "brier": "{:.4f}", "p_media": "{:.1%}", "n_params": "{:,.0f}"}) \
    .highlight_max(subset=["roc_auc", "pr_auc"], color="#d6e6f8").highlight_min(subset=["brier"], color="#d6e6f8")

In [ ]:
from sklearn.metrics import roc_auc_score
ano = df.appointment_date.dt.year
teste = df[ano >= 2021]
ok = teste.taxa_faltas_anteriores.notna()
print(f"Registros com ID reconstruído (histórico disponível): treino ≤2020 {df[ano <= 2020].patient_id.notna().mean():.0%} "
      f"→ teste 2021–2022 {teste.patient_id.notna().mean():.0%}")
print(f"ROC-AUC de taxa_faltas_anteriores sozinha no teste, só onde existe histórico: "
      f"{roc_auc_score(teste.no_show[ok], teste.taxa_faltas_anteriores[ok]):.3f}")

**Leitura da validação temporal:**

- **O histórico generaliza, o calendário não.** Com BIC, o Manto da engenharia é só `taxa_faltas_anteriores`, e é a
  única configuração que se sustenta no futuro. Com AIC, a entrada de `mes` (12 estados) derruba o desempenho: o
  padrão mensal de 2016–2020, confundido com lacunas de cobertura e com a pandemia, não se repete. A escolha do AIC
  feita no notebook 03 **não sobrevive** ao teste no futuro.
- **Os dados brutos não generalizam.** Sem `appointment_year`, o controle C0 depende de mês e horário e fica perto
  do acaso.
- **A qualidade do cadastro limita o modelo.** Em 2021–2022, uma fatia muito maior dos agendamentos não tem ID e,
  portanto, não tem histórico. Onde o histórico existe, ele discrimina bem melhor que o modelo como um todo.

In [ ]:
prev_tr = df[df.appointment_date.dt.year <= 2020].no_show.mean()
prev_te = df[df.appointment_date.dt.year >= 2021].no_show.mean()
print(f"Prevalência de no-show: treino (≤2020) {prev_tr:.1%} → teste (2021–2022) {prev_te:.1%}")
print("Modelos sem SMOTENC preveem em média a prevalência do passado: a deriva aparece como subestimação do risco.")

## 6. Síntese

### Veredito das hipóteses

| Hipótese | Resultado | Evidência |
|---|---|---|
| **H3** (escalonamento + SMOTENC superam os dados brutos em ROC-AUC e PR-AUC) | **Refutada** | Todas as configurações com SMOTENC ficam **abaixo** dos dados brutos e da engenharia sem SMOTENC, nos **15 de 15 folds** (*p* = 6 × 10⁻⁵, significativo após Holm). ROC-AUC cai de 0,632 (C1) para 0,560–0,596 e PR-AUC de 0,159 para 0,117–0,134. O mesmo padrão aparece na validação temporal. |
| **H1** (o framework melhora acurácia **e** Recall) | **Refutada** | No limiar de 0,5, o SMOTENC eleva o Recall (0,3% → 43–48%), mas a acurácia cai de 90% para 62–66%, porque o modelo passa a alertar em 32–37% dos agendamentos. O ganho de Recall vem do volume de alertas, e não de melhor discriminação: um modelo sem SMOTENC com limiar na prevalência detecta 58–69% das faltas, com acurácia balanceada **maior** (0,60–0,63 contra 0,55–0,56). |
| **Escalonadores** | **Efeito nulo ou indireto** | Sem SMOTENC, o efeito é nulo por construção: as probabilidades são idênticas (Seção 1). Com SMOTENC, o **RobustScaler** é o menos prejudicial (ROC-AUC 0,590 contra 0,560 do MinMax), coerente com a fundamentação (Seção 2.3.2): ele limita a influência dos outliers na distância usada para escolher vizinhos. Mas nenhum escalonador supera a ausência de SMOTENC. |

### Por que o SMOTENC falha aqui

1. **Ele deixa uma "assinatura" que o modelo aprende.** Em todos os folds com escalonador, uma variável de **chuva**
   entra no Manto de Markov; com os dados reais, isso nunca acontece. O SMOTENC interpola a chuva numérica e copia
   a categoria de chuva dos vizinhos, gerando combinações que **não existem em nenhum registro real**, e 67% dos
   sintéticos têm um par temperatura-chuva que nenhum dia real teve. Como todo sintético é no-show, essas combinações
   têm 100% de no-show no treino: a rede aprende a reconhecer registros fabricados, um "padrão" ausente no teste.
2. **Ele distorce as probabilidades.** O modelo aprende que ~50% dos agendamentos são faltas e prevê risco médio de
   ~43%, quando a realidade é 10%. O Brier mais que dobra (0,086 → 0,21–0,23). Para uma mensageria que age sobre a
   probabilidade, isso é um defeito grave.
3. **A Rede Bayesiana não precisa de balanceamento para ordenar.** O desbalanceamento afeta a priori P(no-show), não
   a capacidade de ordenar pacientes por risco. Ajustar o **limiar de decisão** resolve o problema de "o modelo nunca
   alerta" sem degradar a estrutura.

### A inversão entre validação aleatória e temporal

| | Validação cruzada aleatória | Validação temporal (treino ≤2020 → teste 2021–22) |
|---|---|---|
| C0 bruto | **0,671** (melhor) | 0,509 (≈ acaso) |
| C1 engenharia | 0,632 | **0,570** (melhor, com BIC) |

Na validação aleatória, os dados brutos vencem porque `appointment_year` (7 estados) captura a deriva temporal com
mais detalhe do que `periodo`. Isso só funciona porque o teste mistura os mesmos anos do treino. Prevendo o futuro,
o ano não ajuda, e o controle bruto cai para perto do acaso, enquanto o **histórico do paciente**, criado pela
engenharia de variáveis, é o único sinal que se sustenta.

**O componente do framework que agrega valor é a engenharia de variáveis** (histórico, correções, ausência como
categoria). E esse valor só aparece numa validação que respeita o tempo.

### Implicações para o texto do TCC

1. **Reportar as hipóteses como refutadas.** É uma contribuição legítima: mostra, com experimento controlado e teste
   estatístico, que a receita "normalizar + SMOTE" não se transfere para Redes Bayesianas discretas em dados de
   agendamento. As Seções 2.3.2 e 2.3.3 precisam ser revisadas: o escalonamento não altera CPTs, e a afirmação de
   que o SMOTENC "neutraliza o viés" sem custo não se confirmou.
2. **Incluir a validação temporal na metodologia (Seção 3.4)** como o critério de generalização mais honesto, ao
   lado da validação cruzada agrupada por paciente. Ela também mudou a escolha do critério de estrutura: **BIC**
   generaliza melhor que AIC, revertendo a preferência do notebook 03.
3. **Reformular a contribuição do framework:** em vez de "normalização + balanceamento melhoram o modelo", a
   contribuição passa a ser um pipeline que **evita vazamento** (histórico causal, folds por paciente, ajuste só no
   treino), **corrige a base** (idade, clima, ausências) e entrega um **modelo parcimonioso, calibrado e auditável**.
   O tratamento do desbalanceamento é feito pelo limiar de decisão.